In [0]:
%pip install -U databricks-langchain

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)

In [0]:
def input_guardrail(user_input: str):
    blocked_words = [
        "password",
        "credit card",
        "hack",
        "malware",
        "bomb",
        "gun"
        "robbery",
    ]

    text = user_input.lower()

    for word in blocked_words:
        if word in text:
            raise ValueError(
                f"Input blocked by guardrail: '{word}' is not allowed."
            )

    return True


user_input = input("Enter your prompt: ")



try:
    # -------------------------
    # INPUT GUARDRAIL
    # -------------------------
    input_guardrail(user_input)

    print("✅ Input passed guardrail")

    # -------------------------
    # LLM
    # -------------------------
    response = llm.invoke(user_input)

    print("\nLLM Response:")
    print(response.content)

except ValueError as e:
    print(f"\n❌ {e}")


In [0]:
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)

conversation_history = []

In [0]:
def chat(user_input):

    # Add user message
    conversation_history.append(
        HumanMessage(content=user_input)
    )

    # Send entire conversation to the model
    response = llm.invoke(conversation_history)

    # Store assistant response
    conversation_history.append(
        AIMessage(content=response.content)
    )

    return response.content

In [0]:
print(chat("My name is Naval."))

In [0]:
print(chat("What is my name?"))

In [0]:
print(conversation_history)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dev.bronze.demo_user_memory (
    user_id STRING,
    memory_key STRING,
    memory_value STRING,
    updated_at TIMESTAMP
)
""")

In [0]:
user_id = "naval"

def save_memory(user_id, key, value):

    spark.sql(f"""
        MERGE INTO dev.bronze.demo_user_memory AS target
        USING (
            SELECT
                '{user_id}' AS user_id,
                '{key}' AS memory_key,
                '{value}' AS memory_value,
                current_timestamp() AS updated_at
        ) AS source

        ON target.user_id = source.user_id
        AND target.memory_key = source.memory_key

        WHEN MATCHED THEN UPDATE SET
            memory_value = source.memory_value,
            updated_at = source.updated_at

        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
save_memory(
    "naval",
    "favorite_language",
    "Python"
)

In [0]:
def get_memories(user_id):

    return spark.sql(f"""
        SELECT memory_key, memory_value
        FROM dev.bronze.demo_user_memory
        WHERE user_id = '{user_id}'
    """).collect()

In [0]:
memories = get_memories("naval")

for memory in memories:
    print(memory.memory_key, "=", memory.memory_value)

In [0]:
def chat_with_long_term_memory(user_id, user_input):

    memories = get_memories(user_id)

    memory_text = "\n".join(
        f"{m.memory_key}: {m.memory_value}"
        for m in memories
    )

    prompt = f"""
You are a helpful AI assistant.

Long-term memory about the user:
{memory_text}

User question:
{user_input}

Use the long-term memory when relevant.
"""

    response = llm.invoke(prompt)

    return response.content

In [0]:
print(
    chat_with_long_term_memory(
        "naval",
        "What programming language do I prefer?"
    )
)

In [0]:
conversation_history.clear()